# BLEU — the copy baseline and zero-shot translation

**Two different BLEUs, two different questions.**

| BLEU | Question it answers |
|---|---|
| copy-task BLEU (2,000 sentences) | how much of the input survives the prompt |
| zero-shot translation BLEU (100 pairs) | what the methodology has to beat, with no training |

Directions are assigned so that the target language decides the model: `en->vi` uses ViT5, `vi->en` uses
Flan-T5 — one base pair, one large pair, never mixed.

In [ ]:
import csv, json, os, sys, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TQDM_DISABLE", "1")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name != "research-prototype" else Path.cwd()
sys.path.insert(0, str(ROOT))
import torch
from src import config as C, models as M, data as D

def read_csv(name):
    p = ROOT / "results" / name
    if not p.exists():
        print(f"missing {p.name} - run: python -m src.run_all (stage not executed yet)")
        return []
    return list(csv.DictReader(p.open(encoding="utf-8")))

print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


## 1. The reference set

In [ ]:
pairs = D.build(force=False)
print("pairs:", len(pairs))
for r in pairs[:3]:
    print(f"[{r['id']}] VI: {r['vi']}")
    print(f"      EN: {r['en']}")

## 2. Zero-shot translation — examples

In [ ]:
raw = ROOT / "results" / "raw"
for f in sorted(raw.glob("tr_*.jsonl")):
    rows = [json.loads(l) for l in f.read_text(encoding="utf-8").splitlines() if l.strip()][:2]
    print("==", f.stem)
    for r in rows:
        p = pairs[r["id"] - 1]
        print("  hyp:", r["hyp"][:110])
        print("  ref:", (p["vi"] if "en2vi" in f.stem else p["en"])[:110])
        print()

## 3. Scores

In [ ]:
table = read_csv("bleu_summary.csv")
cols = ["model", "direction", "n", "bleu", "chrf", "dish_name_kept", "empty_rate", "len_ratio"]
if table:
    print(" | ".join(cols))
    for r in table:
        print(" | ".join(str(r[c]) for c in cols))

`dish_name_kept` is the paper-relevant number that BLEU cannot see: the fraction of outputs that still
contain the dish name (transliterated for `en->vi`, romanised for `vi->en`). A model can score BLEU 0 and
still keep every dish name — which is exactly the cultural-preservation question.

## 4. What this baseline means

In [ ]:
for r in read_csv("bleu_summary.csv"):
    print(f"{r['model']:<14} {r['direction']}  BLEU {r['bleu']:>6}  ->  the mapping must beat this with NO decoder fine-tuning")

**Reading.** A T5-family base checkpoint has no instruction-tuned translation ability, so a low zero-shot
BLEU is expected and is *not* a failure of the methodology — it is the floor. The copy-task BLEU is the
ceiling for "the model can reproduce text at all"; the gap between the two is the work the mapping has to do.